In [11]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully!")

Libraries imported successfully!


In [12]:
df = pd.read_parquet(
    "../data/raw_data/daily_weather.parquet",
    columns=[
        "city_name",
        "date",
        "season",
        "precipitation_mm",
        "avg_wind_speed_kmh",
        "avg_sea_level_pres_hpa",
        "avg_temp_c"
    ]
)

print("Dataset loaded successfully!")
print("Original Shape:", df.shape)

Dataset loaded successfully!
Original Shape: (27635763, 7)


In [13]:
model_sample = df.sample(
    n=200000,
    random_state=42
).copy()

print("Sample Shape:", model_sample.shape)

Sample Shape: (200000, 7)


In [14]:
# Remove rows where city or target temperature is missing

model_sample = model_sample.dropna(
    subset=["city_name", "avg_temp_c"]
).copy()

# Fill numerical missing values with median

numeric_columns = [
    "precipitation_mm",
    "avg_wind_speed_kmh",
    "avg_sea_level_pres_hpa"
]

for column in numeric_columns:
    model_sample[column] = model_sample[column].fillna(
        model_sample[column].median()
    )

print("Missing values handled successfully!")
print(model_sample.isnull().sum())

Missing values handled successfully!
city_name                 0
date                      0
season                    0
precipitation_mm          0
avg_wind_speed_kmh        0
avg_sea_level_pres_hpa    0
avg_temp_c                0
dtype: int64


In [15]:
model_sample["year"] = model_sample["date"].dt.year
model_sample["month"] = model_sample["date"].dt.month
model_sample["day"] = model_sample["date"].dt.day
model_sample["day_of_year"] = model_sample["date"].dt.dayofyear

model_sample = model_sample.drop(
    columns=["date"]
)

print("Date features created successfully!")

Date features created successfully!


In [16]:
# Convert categorical columns into numerical codes

model_sample["city_code"] = (
    model_sample["city_name"].astype("category").cat.codes
)

model_sample["season_code"] = (
    model_sample["season"].astype("category").cat.codes
)

# Remove original categorical columns

model_sample = model_sample.drop(
    columns=["city_name", "season"]
)

print("Categorical encoding completed!")
print(model_sample.head())

Categorical encoding completed!
       precipitation_mm  avg_wind_speed_kmh  avg_sea_level_pres_hpa  \
1503                0.0                10.9                  1014.7   
23578               4.5                10.9                  1014.7   
8609                0.3                10.9                  1014.7   
17783               0.0                 9.6                  1009.2   
2594                0.0                10.9                  1014.7   

       avg_temp_c  year  month  day  day_of_year  city_code  season_code  
1503         25.7  2008      7   27          209        549            2  
23578        14.1  1965      7   22          203       1009            2  
8609         12.3  1989      2    1           32        258            3  
17783        29.0  2019      7   23          204        210            2  
2594         26.4  1968      5    1          122        595            0  


In [17]:
X = model_sample.drop(
    columns=["avg_temp_c"]
)

y = model_sample["avg_temp_c"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

X Shape: (155070, 9)
y Shape: (155070,)

Features:
['precipitation_mm', 'avg_wind_speed_kmh', 'avg_sea_level_pres_hpa', 'year', 'month', 'day', 'day_of_year', 'city_code', 'season_code']


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (124056, 9)
Testing Shape: (31014, 9)


In [19]:
model = RandomForestRegressor(
    n_estimators=30,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

print("Random Forest model created successfully!")

Random Forest model created successfully!


In [20]:
print("Model training started...")

model.fit(
    X_train,
    y_train
)

print("Model training completed successfully!")

Model training started...
Model training completed successfully!


In [21]:
print("Making predictions...")

y_pred = model.predict(X_test)

print("Predictions completed successfully!")
print("Number of predictions:", len(y_pred))

Making predictions...
Predictions completed successfully!
Number of predictions: 31014


In [22]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

r2 = r2_score(
    y_test,
    y_pred
)

print("Model Evaluation Results")
print("------------------------")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 2))

Model Evaluation Results
------------------------
MAE : 6.41
RMSE: 8.42
R²  : 0.51


In [23]:
comparison = pd.DataFrame({
    "Actual Temperature": y_test.values,
    "Predicted Temperature": y_pred
})

comparison["Difference"] = (
    comparison["Actual Temperature"]
    - comparison["Predicted Temperature"]
)

comparison.head(10)

,Actual Temperature,Predicted Temperature,Difference
0,29.2,25.700507,3.499493
1,14.8,11.110425,3.689575
2,27.2,24.949190,2.250810
3,5.8,14.887442,-9.087442
4,10.8,7.622825,3.177175
5,24.1,9.630751,14.469249
6,21.2,22.204791,-1.004791
7,17.3,13.497459,3.802541
8,16.1,15.339109,0.760891
9,18.3,24.361248,-6.061248


In [24]:
joblib.dump(
    model,
    "../models/weather_model.pkl"
)

print("Weather model saved successfully!")

Weather model saved successfully!


In [25]:
feature_columns = X.columns.tolist()

joblib.dump(
    feature_columns,
    "../models/feature_columns.pkl"
)

print("Feature columns saved successfully!")
print(feature_columns)

Feature columns saved successfully!
['precipitation_mm', 'avg_wind_speed_kmh', 'avg_sea_level_pres_hpa', 'year', 'month', 'day', 'day_of_year', 'city_code', 'season_code']


In [26]:
print("===================================")
print("WEATHER FORECASTER - FINAL SUMMARY")
print("===================================")

print("Training Rows :", len(X_train))
print("Testing Rows  :", len(X_test))
print("Features      :", len(X.columns))

print("\nModel: Random Forest Regressor")

print("\nPerformance:")
print("MAE :", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²  :", round(r2, 2))

print("\nModel saved to:")
print("../models/weather_model.pkl")

print("\nTraining completed successfully!")

WEATHER FORECASTER - FINAL SUMMARY
Training Rows : 124056
Testing Rows  : 31014
Features      : 9

Model: Random Forest Regressor

Performance:
MAE : 6.41
RMSE: 8.42
R²  : 0.51

Model saved to:
../models/weather_model.pkl

Training completed successfully!


In [28]:
# Recreate city and season mappings used during training

import pandas as pd
import joblib

# Load only the columns needed for the mappings
mapping_df = pd.read_parquet(
    "../data/raw_data/daily_weather.parquet",
    columns=[
        "city_name",
        "date",
        "season",
        "precipitation_mm",
        "avg_wind_speed_kmh",
        "avg_sea_level_pres_hpa",
        "avg_temp_c"
    ]
)

# Use the exact same sample
mapping_df = mapping_df.sample(
    n=200000,
    random_state=42
).copy()

# Remove rows where city or target temperature was missing
mapping_df = mapping_df.dropna(
    subset=["city_name", "avg_temp_c"]
).copy()

# Create the same city categories used during training
city_categories = (
    mapping_df["city_name"]
    .astype("category")
    .cat.categories
    .tolist()
)

# Create the same season categories used during training
season_categories = (
    mapping_df["season"]
    .astype("category")
    .cat.categories
    .tolist()
)

# Save mappings
joblib.dump(
    city_categories,
    "../models/city_categories.pkl"
)

joblib.dump(
    season_categories,
    "../models/season_categories.pkl"
)

print("City mapping saved successfully!")
print("Season mapping saved successfully!")

print("Number of cities:", len(city_categories))
print("Seasons:", season_categories)

City mapping saved successfully!
Season mapping saved successfully!
Number of cities: 1234
Seasons: ['Autumn', 'Spring', 'Summer', 'Winter']
